[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/17_feature_engineering/90_exercises.ipynb)

# Bài tập — Chương 17: Feature Engineering (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/17_feature_engineering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Liệt kê: 3 biến đổi cho feature lệch; 5 cách mã hoá category; công thức target encoding có smoothing; công thức mã hoá chu kỳ cho giờ.

✍️ Trả lời:

**Đáp án:** log / log1p, Box-Cox / Yeo-Johnson, quantile transform. Category: one-hot, ordinal, frequency/count, target encoding (cross-fit + smoothing), hashing (+ native, embedding). Smoothing: $\hat\mu_c = \frac{n_c\bar y_c + m\bar y}{n_c + m}$. Giờ: $(\sin\frac{2\pi h}{24}, \cos\frac{2\pi h}{24})$.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích cho một người mới vì sao "trung bình số tiền giao dịch của khách hàng" tính trên **toàn bộ** dữ liệu là feature nguy hiểm khi phát hiện gian lận, và cách tính đúng.

✍️ Trả lời:

**Gợi ý:** trung bình trên toàn bộ dữ liệu bao gồm cả giao dịch **tương lai** và **chính** giao dịch đang xét — thông tin không có tại thời điểm cần ra quyết định. Giao dịch gian lận lớn bất thường làm trung bình của khách hàng đó tăng lên, nên feature "biết trước" có gian lận → model học được trên dữ liệu lịch sử nhưng thất bại khi triển khai. Cách đúng: chỉ dùng giao dịch **trước** thời điểm hiện tại (cumulative mean trừ chính nó, rolling window đóng bên trái), và trong CV theo thời gian tính lại cho từng fold.

## L2 — Tính tay

**Câu 3.** Target encoding có smoothing $m = 10$, tỉ lệ chung $\bar y = 0.2$. Ba category: A (100 mẫu, 30 positive), B (4 mẫu, 4 positive), C (0 mẫu ở train). Tính encoding của A, B, C.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
enc_A = (100 * 0.30 + 10 * 0.2) / (100 + 10)
enc_B = (4 * 1.0 + 10 * 0.2) / (4 + 10)
enc_C = 0.2
### END CODE HERE ###

In [ ]:
check_close(enc_A, 32 / 110, message="A")
check_close(enc_B, 6 / 14, message="B — bị kéo mạnh về tỉ lệ chung")
check_close(enc_C, 0.2, message="C — category mới = tỉ lệ chung")

**Câu 4.** Mã hoá chu kỳ cho tháng (1–12): tính $(\sin, \cos)$ của tháng 12 và tháng 1, và khoảng cách Euclid giữa chúng; so với khoảng cách giữa tháng 6 và tháng 7, và giữa 12 và 1 khi dùng số tháng thô.

In [ ]:
def cyc(m):
    return np.array([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)])


### START CODE HERE ### (≈ 3 dòng)
d_12_1 = np.linalg.norm(cyc(12) - cyc(1))
d_6_7 = np.linalg.norm(cyc(6) - cyc(7))
d_raw_12_1 = abs(12 - 1)
### END CODE HERE ###

In [ ]:
check_close(d_12_1, d_6_7, message="mã hoá chu kỳ: 12→1 cách đều như 6→7")
check_close(d_12_1, 2 * np.sin(np.pi / 12), message="= 2·sin(π/12)")
check_equal(d_raw_12_1, 11, "số thô: 12 và 1 cách 11")

## L3 — Code

**Bài 5.** Viết `CyclicalEncoder(period)` — transformer scikit-learn: với mỗi cột, trả về hai cột sin và cos; có `get_feature_names_out`. Dùng trong Pipeline dự đoán nhu cầu theo giờ.

In [ ]:
class CyclicalEncoder(TransformerMixin, BaseEstimator):
    def __init__(self, period=24):
        self.period = period

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        ### START CODE HERE ### (≈ 3 dòng)
        # STARTER: X = None        # mảng float
        # STARTER: angle = None    # 2π·X / period
        # STARTER: return None     # np.hstack([sin(angle), cos(angle)])
        X = np.asarray(X, float)
        angle = 2 * np.pi * X / self.period
        return np.hstack([np.sin(angle), np.cos(angle)])
        ### END CODE HERE ###

    def get_feature_names_out(self, input_features=None):
        names = input_features if input_features is not None else [f"x{i}" for i in range(self.n_features_in_)]
        return np.array([f"{n}_sin" for n in names] + [f"{n}_cos" for n in names])

In [ ]:
hours = rng.integers(0, 24, 3000)
demand = 50 + 20 * np.cos(2 * np.pi * (hours - 18) / 24) + rng.normal(0, 3, 3000)
r2_raw = cross_val_score(LinearRegression(), hours[:, None], demand, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
r2_cyc = cross_val_score(make_pipeline(CyclicalEncoder(24), LinearRegression()), hours[:, None], demand,
                         cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
print(f"R²: giờ thô = {r2_raw:.3f} | sin/cos = {r2_cyc:.3f}")
check_greater(r2_cyc, 0.9, "mã hoá chu kỳ bắt được nhu cầu theo giờ")
check_equal(list(CyclicalEncoder(24).fit(hours[:, None]).get_feature_names_out(["hour"])), ["hour_sin", "hour_cos"], "tên feature")

**Bài 6.** Viết `smoothed_target_encode_cv(cat, y, m, n_splits, seed)` — target encoding **cross-fit**: với mỗi fold, tính encoding (có smoothing, prior = trung bình y của các fold còn lại) cho các mẫu trong fold từ các fold **khác**. Trả về mảng encoding cho mọi mẫu train.

In [ ]:
def smoothed_target_encode_cv(cat, y, m=10, n_splits=5, seed=0):
    cat, y = np.asarray(cat), np.asarray(y, float)
    out = np.zeros(len(y))
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(cat):
        ### START CODE HERE ### (≈ 5 dòng)
        # STARTER: prior = None    # trung bình y của các fold KHÁC (tr)
        # STARTER: stats = None    # tổng và số mẫu y theo category trên tr
        # STARTER: enc = None      # (sum + m·prior) / (count + m)
        # STARTER: # out[va] = encoding của cat[va] (category chưa thấy → prior)
        prior = y[tr].mean()
        stats = pd.DataFrame({"c": cat[tr], "y": y[tr]}).groupby("c")["y"].agg(["sum", "count"])
        enc = (stats["sum"] + m * prior) / (stats["count"] + m)
        out[va] = pd.Series(cat[va]).map(enc).fillna(prior).to_numpy()
        ### END CODE HERE ###
    return out

In [ ]:
cats = rng.integers(0, 300, 6000).astype(str)
effect = dict(zip(np.unique(cats), rng.normal(0, 1, 300)))
y6 = (rng.random(6000) < 1 / (1 + np.exp(-(-1 + np.array([effect[c] for c in cats]))))).astype(int)
enc_cv = smoothed_target_encode_cv(cats, y6)
naive = pd.Series(y6).groupby(cats).transform("mean").to_numpy()
print(f"ROC-AUC của chính cột encoding trên train: ngây thơ = {roc_auc_score(y6, naive):.3f} | cross-fit = {roc_auc_score(y6, enc_cv):.3f}")
check_less(roc_auc_score(y6, enc_cv), roc_auc_score(y6, naive) - 0.03, "cross-fit không 'nhìn thấy' nhãn của chính mẫu")
check_greater(roc_auc_score(y6, enc_cv), 0.6, "nhưng vẫn mang tín hiệu thật")

## L4 — Debug

**Bài 7.** Pipeline dưới đây có **3 lỗi**: (1) log-transform áp dụng cho feature có giá trị 0 (`n_tx_last_24h`) → `-inf`; (2) `StandardScaler` và target encoding thủ công được tính trên toàn bộ dữ liệu trước khi chia; (3) feature `amount_vs_customer_mean` dùng trung bình của khách hàng tính trên **toàn bộ** dữ liệu. Sửa: dùng `np.log1p`, đưa scaler vào Pipeline, và tính feature tổng hợp **chỉ từ quá khứ**. Lưu ROC-AUC test vào `test_auc`.

In [ ]:
fraud = pd.read_csv(repo_path("data", "sample", "fraud_transactions.csv"))
fraud = fraud.reset_index().rename(columns={"index": "t"})                       # thứ tự dòng = thời gian
fraud["customer"] = np.random.default_rng(1).integers(0, 1500, len(fraud))       # gán khách hàng (minh hoạ)
### START CODE HERE ### (≈ 10 dòng)
# STARTER: fraud["log_tx"] = np.log(fraud["n_tx_last_24h"])                                                   # 🐛 (1)
# STARTER: fraud["amount_vs_customer_mean"] = fraud["amount_vnd"] / fraud.groupby("customer")["amount_vnd"].transform("mean")   # 🐛 (3)
# STARTER: feats = ["amount_vnd", "log_tx", "distance_from_home_km", "amount_vs_customer_mean"]
# STARTER: X_all = StandardScaler().fit_transform(fraud[feats])                                                # 🐛 (2)
# STARTER: X_tr, X_te, y_tr, y_te = train_test_split(X_all, fraud["is_fraud"], test_size=0.3, shuffle=False)
# STARTER: test_auc = roc_auc_score(y_te, LogisticRegression(max_iter=2000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1])
fraud["log_tx"] = np.log1p(fraud["n_tx_last_24h"])
g = fraud.groupby("customer")["amount_vnd"]
past_mean = (g.cumsum() - fraud["amount_vnd"]) / g.cumcount().replace(0, np.nan)
fraud["amount_vs_customer_mean"] = (fraud["amount_vnd"] / past_mean).fillna(1.0)
feats = ["amount_vnd", "log_tx", "distance_from_home_km", "amount_vs_customer_mean"]
X_tr, X_te, y_tr, y_te = train_test_split(fraud[feats], fraud["is_fraud"], test_size=0.3, shuffle=False)
model7 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(np.log1p(X_tr), y_tr)
test_auc = roc_auc_score(y_te, model7.predict_proba(np.log1p(X_te))[:, 1])
### END CODE HERE ###
print(f"ROC-AUC test = {test_auc:.3f}")

In [ ]:
check_true(np.isfinite(fraud["log_tx"]).all(), "không còn -inf")
check_true(fraud.loc[fraud.groupby("customer").cumcount() == 0, "amount_vs_customer_mean"].eq(1.0).all(),
           "giao dịch đầu tiên của mỗi khách không có quá khứ → giá trị mặc định")
check_greater(test_auc, 0.7, "ROC-AUC test > 0.7")

## L5 — Phân tích

**Bài 8.** Trên `house_prices.csv`, đo ảnh hưởng của từng biến đổi lên **Linear Regression** và **HistGradientBoosting** (5-fold MAE): (a) feature gốc; (b) + log(area) và log target; (c) + area_per_bedroom. Lưu DataFrame `ablation` (hàng = cấu hình, cột = model). Nhận xét: biến đổi nào giúp model nào, vì sao.

In [ ]:
from sklearn.compose import TransformedTargetRegressor

houses = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
H = houses.drop(columns="price_billion_vnd").assign(has_garden=lambda d: d.has_garden.astype(float),
                                                    log_area=lambda d: np.log(d.area_m2),
                                                    area_per_bedroom=lambda d: d.area_m2 / d.bedrooms)
yh = houses["price_billion_vnd"]
base_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km", "has_garden"]
configs = {"(a) gốc": (base_cols, False), "(b) + log(area), log target": (base_cols + ["log_area"], True),
           "(c) + area_per_bedroom": (base_cols + ["log_area", "area_per_bedroom"], True)}
rows = {}
### START CODE HERE ### (≈ 9 dòng)
# STARTER: # với mỗi cấu hình và mỗi model (LinearRegression, HistGradientBoostingRegressor): Pipeline(ColumnTransformer(StandardScaler cột số, OneHotEncoder district), model),
# STARTER: # bọc TransformedTargetRegressor(log/exp) nếu log_t; rows[tên cấu hình][tên model] = MAE 5-fold (KFold shuffle, random_state=0)
for name, (cols, log_t) in configs.items():
    rows[name] = {}
    for model_name, model in [("Linear", LinearRegression()), ("HistGB", HistGradientBoostingRegressor(random_state=0))]:
        pre = ColumnTransformer([("num", StandardScaler(), cols), ("cat", OneHotEncoder(), ["district"])])
        est = make_pipeline(pre, model)
        if log_t:
            est = TransformedTargetRegressor(est, func=np.log, inverse_func=np.exp)
        rows[name][model_name] = -cross_val_score(est, H, yh, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error").mean()
### END CODE HERE ###
ablation = pd.DataFrame(rows).T
ablation.round(3)

In [ ]:
check_less(ablation.loc["(b) + log(area), log target", "Linear"], ablation.loc["(a) gốc", "Linear"] - 0.05, "log–log giúp Linear nhiều")
check_less(abs(ablation.loc["(b) + log(area), log target", "HistGB"] - ablation.loc["(a) gốc", "HistGB"]), 0.06,
           "HistGB ít thay đổi với biến đổi đơn điệu")

✍️ Nhận xét:

**Gợi ý:** log–log biến quan hệ nhân (giá ∝ diện tích × hệ số quận) thành cộng → Linear Regression cải thiện lớn. HistGB bất biến với biến đổi đơn điệu của **feature** (log_area chỉ là bản sao đơn điệu của area) nên gần như không đổi; log **target** đổi hàm loss hiệu dụng (sai số tương đối) nên có thể thay đổi chút ít. Feature tỉ số có thể giúp cả hai vì cây khó tự học phép chia.

## L6 — Thực tế

**Bài 9.** Model churn tốt nhất có thể với **Logistic Regression** (model dễ giải thích cho đội kinh doanh) trên `customer_churn.csv` bằng feature engineering: gợi ý — log(total_charges + 1), `monthly_charges / (tenure + 1)`, tương tác `contract × tenure`, spline cho tenure, one-hot các cột phân loại. So sánh ROC-AUC (5-fold) với LR trên feature gốc (`base_auc`); yêu cầu `fe_auc` không thấp hơn `base_auc` và trả lời câu hỏi bên dưới.

In [ ]:
from sklearn.preprocessing import SplineTransformer

churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
yc = churn["churn"].to_numpy()
cat_cols = ["contract", "internet_service", "payment_method"]
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
base = ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(), cat_cols)])
cv9 = StratifiedKFold(5, shuffle=True, random_state=0)
base_auc = cross_val_score(make_pipeline(base, LogisticRegression(max_iter=3000)), churn, yc, cv=cv9, scoring="roc_auc").mean()
### START CODE HERE ### (≈ 12 dòng)
fe = churn.assign(log_total=np.log1p(churn.total_charges),
                  charge_per_tenure=churn.monthly_charges / (churn.tenure_months + 1),
                  m2m_tenure=(churn.contract == "month-to-month") * churn.tenure_months,
                  senior=churn.senior.astype(float))
fe_cols = num_cols + ["log_total", "charge_per_tenure", "m2m_tenure", "senior"]
pre = ColumnTransformer([("num", StandardScaler(), fe_cols),
                         ("spline", SplineTransformer(n_knots=5), ["tenure_months"]),
                         ("cat", OneHotEncoder(), cat_cols)])
fe_auc = cross_val_score(make_pipeline(pre, LogisticRegression(C=0.5, max_iter=5000)), fe, yc, cv=cv9, scoring="roc_auc").mean()
### END CODE HERE ###
print(f"ROC-AUC (CV): gốc = {base_auc:.4f} | feature engineering = {fe_auc:.4f}")

In [ ]:
check_greater(fe_auc, base_auc - 0.002, "feature engineering không làm tệ đi")

✍️ Mức cải thiện lớn hay nhỏ? Vì sao? (Gợi ý: xem cách `make_customer_churn` sinh nhãn trong `src/ml_course/datasets.py`.)

**Gợi ý:** cải thiện rất nhỏ — vì nhãn được sinh từ một **logit tuyến tính** theo chính các feature gốc (tenure, monthly_charges, support_calls, contract...), nên Logistic Regression trên feature gốc đã gần tối ưu; feature mới chỉ thêm biến thể của thông tin đã có (và `senior` — feature gốc bị bỏ sót). Bài học: feature engineering mang lại nhiều nhất khi quan hệ thật **phi tuyến/tương tác** hoặc khi feature thô chưa phản ánh đúng cấu trúc bài toán (như log–log ở giá nhà). Luôn đo bằng CV — đừng giả định feature mới sẽ giúp.

## L7 — Nâng cao

**Bài 10.** Cài đặt **hashing trick**: `hash_encode(values, n_buckets, seed)` ánh xạ mỗi chuỗi vào một trong `n_buckets` cột (dùng `hashlib.md5` để ổn định giữa các lần chạy) với dấu ±1 (từ một bit khác của hash) — trả về ma trận thưa `scipy.sparse.csr_matrix`. Đo **va chạm** (nhiều category rơi vào cùng bucket) khi băm 1 000 category vào 256 bucket: số bucket được dùng so với kỳ vọng lý thuyết $256\left(1 - (1 - \frac{1}{256})^{1000}\right)$. (scikit-learn có sẵn `FeatureHasher` làm việc này.)

In [ ]:
import hashlib

from scipy import sparse


def hash_encode(values, n_buckets=256, seed=0):
    rows, cols, data = [], [], []
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: # với mỗi giá trị: h = int(md5(f"{seed}:{v}").hexdigest(), 16); cột = h % n_buckets; dấu = ±1 theo bit (h >> 64) & 1
    # STARTER: # thêm (i, cột, dấu) vào rows, cols, data
    for i, v in enumerate(values):
        h = int(hashlib.md5(f"{seed}:{v}".encode()).hexdigest(), 16)
        rows.append(i)
        cols.append(h % n_buckets)
        data.append(1.0 if (h >> 64) & 1 else -1.0)
    ### END CODE HERE ###
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(values), n_buckets))

In [ ]:
cats10 = [f"merchant_{i}" for i in range(1000)]
H10 = hash_encode(cats10, 256)
buckets = H10.indices                                                    # mỗi hàng có đúng 1 phần tử khác 0
n_used = len(np.unique(buckets))
collision_rate = 1 - n_used / len(cats10)
expected_used = 256 * (1 - (1 - 1 / 256) ** 1000)
print(f"{len(cats10)} category → {n_used} bucket được dùng | tỉ lệ 'mất' do va chạm = {collision_rate:.3f} "
      f"(kỳ vọng ≈ {1 - expected_used / 1000:.3f})")
check_equal(H10.shape, (1000, 256), "shape")
check_true(set(np.unique(H10.data)) <= {-1.0, 1.0}, "giá trị ±1")
check_close(n_used, expected_used, tol=0.05 * expected_used, message="số bucket dùng ≈ kỳ vọng lý thuyết")
check_true(np.array_equal(hash_encode(["a", "b"], 16).toarray(), hash_encode(["a", "b"], 16).toarray()), "ổn định giữa các lần gọi")